# JSON format in Python
Horacio Gomez-Acevedo

BMIG 50303

UAMS

**References:** 
https://realpython.com/python-json/ (conceptual reference; exercises original).
https://docs.python.org/3/library/json.html/ (official JSON doc for python).


## Introduction

JSON (JavaScript Object Notation) is a language-independent text format for structured data. JSON documents can contain objects (`{}`), arrays (`[]`), strings, numbers, booleans, and `null`. 

JSON has some rather strict rules

+ Double-quoted keys and strings
+ No comments or trailing commas 
+ Keys must be strings
+ Booleans are lowercase. 
+ Top-level JSON may be any JSON value, although objects/arrays are common.

Some other key points are that
+ The term “object” in the context of JSON processing in Python can be ambiguous. All values in Python are objects. In JSON, an object refers to any data wrapped in curly braces, similar to a Python dictionary.
+ JSON is not an API or a database.




In [1]:
import json
json_text = '{"gene": "BRCA1", "chromosome": 17, "reviewed": true, "missing": null}'
record = json.loads(json_text)
print(type(json_text), type(record), record)

<class 'str'> <class 'dict'> {'gene': 'BRCA1', 'chromosome': 17, 'reviewed': True, 'missing': None}


### Basic loading conversion

| JSON | Python `loading` |
|---|---|
| object | dict |
| array | list |
| string | str |
| number | int or float |
| true/false | True/False |
| null | None |


The next chunk of code has dictionaries inside dictionaries. The typical extraction in `python style` would be

In [2]:
sample = {"gene": {"symbol": "BRCA1", "taxon": {"name": "Homo sapiens"}}, "variants": [{"id": "v1", "position": 100}, {"id": "v2", "position": 250}], "curated": True}
print(sample["gene"]["taxon"]["name"])
# TODO: print first variant position
# TODO: create list of variant IDs

Homo sapiens


### Class exercise
+ How to print the first variant position?
+ How to create a list of variant IDs?

## Serialization and deserialization

**Serialization** means to convert objects from Python into JSON and **desearialization** is the opposite, meaning to send JSON objects into Python.


There are some functions that are worth describing

| Function | Input | Example of use |
|---|---|---|
|`json.load()` | File object | Reading JSON from a file | 
|`json.loads()` | String containing JSON | Reading JSON from a string|



In [3]:
import json

with open("hdata.json", "r") as f:
    data = json.load(f)

print(data)

{'name': 'Horacio', 'department': 'DBMI'}


In [5]:
import json

json_string = '{"name": "Horacio", "department": "DBMI"}'

data = json.loads(json_string)

print(data)

{'name': 'Horacio', 'department': 'DBMI'}


On the other hand

|Function | Output|
|---|---|
|`json.dump()` | Write JSON to a file|
|`json.dumps()` | Conversion of a Python object into a JSON string|


> JSON is not automatically able to encode `set`, `datetime`, or NumPy values.

In [6]:
text = json.dumps(sample, indent=2, ensure_ascii=False, sort_keys=True)
print(text)
restored = json.loads(text)
assert restored == sample

{
  "curated": true,
  "gene": {
    "symbol": "BRCA1",
    "taxon": {
      "name": "Homo sapiens"
    }
  },
  "variants": [
    {
      "id": "v1",
      "position": 100
    },
    {
      "id": "v2",
      "position": 250
    }
  ]
}


## File I/O and validation

When open a file, it is a good practice to define the encoding. Normally, the UTF-8 is used for maximal interoperability (https://docs.python.org/3/library/json.html#character-encodings)

Next example shows some deficiencies in loading into JSON format.



In [7]:
from pathlib import Path
path = Path("biomedical_record.json")
with path.open("w", encoding="utf-8") as f:
    json.dump(sample, f, indent=2, ensure_ascii=False)
with path.open(encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded["gene"]["symbol"])
for invalid in ["{'gene': 'BRCA1'}", '{"a": 1,}', '{"active": True}']:
    try: json.loads(invalid)
    except json.JSONDecodeError as exc: print("Invalid:", exc.msg)

BRCA1
Invalid: Expecting property name enclosed in double quotes
Invalid: Expecting property name enclosed in double quotes
Invalid: Expecting value


> Note that `json.loads` validates syntax, not whether a record conforms to a biomedical schema.

In [ ]:
publications = {"results": [
    {"pmid": "101", "article": {"title": "Study A", "journal": "Journal X"}, "keywords": ["genomics", "AI"]},
    {"pmid": "102", "article": {"title": "Study B"}, "keywords": []}
    ]}
rows = []
for item in publications["results"]:
    article = item.get("article") or {}
    rows.append({"pmid": item.get("pmid"), "title": article.get("title", ""), "journal": article.get("journal", "Unknown"), "keyword_count": len(item.get("keywords") or [])})
rows

### Class exercise

1. Write rows to publications.json, read back, assert length == 2


## Use of AI
Your instructor used Chatgpt and Copilot to generate some content. All code, text and references were verified for accuracy by your instructor.